In [1]:
import math

def resolucao_teste_completa():
    print("=======================================")
    print("   RESOLUCAO TESTE: VENTO + NEVE")
    print("=======================================")
    print("Nota: Calculos arredondados a 4 casas.\n")

    # ==========================================
    # PARTE 1: VENTO - DADOS E q_p(z)
    # ==========================================
    print("--- 1. VENTO: DADOS DE ENTRADA ---")
    
    try:
        h_coluna = float(input("Altura do pilar (h) [m]: "))
        L_vao = float(input("Vao do portico (L) [m]: "))
        angulo_graus = float(input("Inclinacao cobertura [graus]: "))
    except ValueError:
        print("Erro: Valores invalidos.")
        return

    zona = input("Zona de Vento (A ou B): ").upper().strip()
    if zona == 'B':
        v_b0 = 30.0
    else:
        v_b0 = 27.0
    


    altura_triangulo = (L_vao / 2) * math.tan(math.radians(angulo_graus))
    z_cumeeira = round(h_coluna + altura_triangulo, 4)
    
    vb = round(1.0 * 1.0 * v_b0, 4)
    
    print("\n---------------------------------------")
    print("RESULTADOS ALINEA 1: q_p(z)")
    print("---------------------------------------")
    print("Alt. Cumeeira (z)  =", z_cumeeira, "m")
    print("vb                 =", vb, "m/s")
    
    print("\nCategorias de Terreno:")
    print("1: Zona I (Mar)")
    print("2: Zona II (Vegetacao)")
    print("3: Zona III (Suburbana)")
    print("4: Zona IV (Urbana)")
    
    try:
        cat_input_str = input("Categoria (1-4): ")
        cat_input = int(cat_input_str)
    except ValueError:
        cat_input = 3
    
    # Dicionario manual para evitar erros de sintaxe complexa
    z0 = 0.3
    z_min = 8.0
    
    if cat_input == 1:
        z0 = 0.005
        z_min = 1.0
    elif cat_input == 2:
        z0 = 0.05
        z_min = 3.0
    elif cat_input == 3:
        z0 = 0.3
        z_min = 8.0
    elif cat_input == 4:
        z0 = 1.0
        z_min = 15.0

    # --- CALCULOS ALINEA 1 ---



    z_calc = z_cumeeira
    if z_min > z_cumeeira:
        z_calc = z_min

    kr = round(0.19 * math.pow((z0 / 0.05), 0.07), 4)
    cr = round(kr * math.log(z_calc / z0), 4)
    vm = round(cr * 1.0 * vb, 4)
    Iv = round(1.0 / (1.0 * math.log(z_calc / z0)), 4)

    qp_kilonewton = ((1 + 7 * Iv) * 0.5 * 1.25 * (vm ** 2)) / 1000

    print("\n---------------------------------------")
    print("RESULTADOS ALINEA 1: q_p(z)")
    print("---------------------------------------")
    print("Parametro z0       =", z0, "m")
    print("Parametro z_min    =", z_min, "m")
    print("Alt. Calculo (z_e) =", z_calc, "m")
    print("Fator Terreno (kr) =", kr)
    print("Rugosidade (cr)    =", cr)
    print("Vel. Media (vm)    =", vm, "m/s")
    print("Turbulencia (Iv)   =", Iv)
    print("--> q_p(z)         =", round(qp_kilonewton, 4), "kN/m2")
    print("---------------------------------------\n")

    # ==========================================
    # PARTE 2: COEFICIENTES DE PRESSAO (c_pe)
    # ==========================================
    print("--- 2. VENTO: COEFICIENTES (c_pe) ---")
    
    try:
        comprimento_edificio = float(input("Comprimento pavilhao (b) [m]: "))
    except ValueError: return

    print("\nDirecao do Vento:")
    print("1: Transversal (0 deg)")
    print("2: Longitudinal (90 deg)")
    try:
        dir_str = input("Escolha (1 ou 2): ")
        dir_input = int(dir_str)
    except ValueError: return

    # Definicao de b, d, h
    if dir_input == 1:
        # Transversal
        b = comprimento_edificio
        d = L_vao
        h = z_cumeeira
        tipo_vento = "Transversal"
        zonas_telhado = ['F', 'G', 'H', 'I', 'J']
        
        # Dados simplificados para evitar estruturas de dados complexas
        # Usamos listas simples para interpolaçao
    else:
        # Longitudinal
        b = L_vao
        d = comprimento_edificio
        h = h_coluna
        tipo_vento = "Longitudinal"
        zonas_telhado = ['F', 'G', 'H', 'I']

    e = b
    if 2*h < b:
        e = 2*h
    
    hd_ratio = h / d

    print("\n---------------------------------------")
    print("GEOMETRIA VENTO:", tipo_vento)
    print("---------------------------------------")
    print("b (perp.) =", b, "m")
    print("d (paral.) =", d, "m")
    print("h (ref.)   =", h, "m")
    print("e          =", round(e, 4), "m")
    print("Razao h/d  =", round(hd_ratio, 4))

    # --- PAREDES ---
    print("\n>>> PAREDES VERTICAIS")
    # Logica de get_cpe_wall inline para evitar erros de escopo
    
    print("Coeficientes c_pe,10 (Paredes):")
    zonas_parede = ['A', 'B', 'C', 'D', 'E']
    
    # Valores fixos Tabela 7.1
    # 5.0:  A:-1.2, B:-0.8, C:-0.5, D:0.8, E:-0.7
    # 1.0:  A:-1.2, B:-0.8, C:-0.5, D:0.8, E:-0.5
    # 0.25: A:-1.2, B:-0.8, C:-0.5, D:0.7, E:-0.3
    
    for z in zonas_parede:
        val = 0.0
        # Definicao dos valores base para a zona atual
        v5 = 0.0
        v1 = 0.0
        v025 = 0.0
        
        if z == 'A': 
            v5, v1, v025 = -1.2, -1.2, -1.2
        elif z == 'B': 
            v5, v1, v025 = -0.8, -0.8, -0.8
        elif z == 'C': 
            v5, v1, v025 = -0.5, -0.5, -0.5
        elif z == 'D': 
            v5, v1, v025 = 0.8, 0.8, 0.7
        elif z == 'E': 
            v5, v1, v025 = -0.7, -0.5, -0.3
            
        # Interpolacao
        if hd_ratio >= 5.0:
            val = v5
        elif hd_ratio <= 0.25:
            val = v025
        elif hd_ratio > 1.0:
            # Entre 1.0 e 5.0
            val = v1 + (hd_ratio - 1.0) * (v5 - v1) / (4.0)
        else:
            # Entre 0.25 e 1.0
            val = v025 + (hd_ratio - 0.25) * (v1 - v025) / (0.75)
            
        print("  Zona", z, ":", round(val, 4))

    # --- COBERTURA ---
    print("\n>>> COBERTURA DUAS AGUAS")
    print("Coeficientes c_pe,10 (Cobertura):")
    
    # Funcao auxiliar para interpolar cobertura dentro do loop
    # para evitar problemas de definicao de funcoes aninhadas na TI
    for z in zonas_telhado:
        # Definir os valores para os angulos chave (5, 15, 30, 45, 60, 75)
        # Baseado nos dicionarios anteriores mas simplificado
        v_angs = {}
        
        if dir_input == 1: # Transversal
            if z == 'F': v_angs = {5:[-1.7, 0.0], 15:[-0.9, 0.2], 30:[-0.5, 0.7], 45:[0.0, 0.7], 60:[0.7], 75:[0.8]}
            elif z == 'G': v_angs = {5:[-1.2, 0.0], 15:[-0.8, 0.2], 30:[-0.5, 0.7], 45:[0.0, 0.7], 60:[0.7], 75:[0.8]}
            elif z == 'H': v_angs = {5:[-0.6, 0.0], 15:[-0.3, 0.2], 30:[-0.2, 0.4], 45:[0.0, 0.6], 60:[0.7], 75:[0.8]}
            elif z == 'I': v_angs = {5:[-0.6, 0.0], 15:[-0.4, 0.0], 30:[-0.4, 0.0], 45:[-0.2, 0.0], 60:[-0.2], 75:[-0.2]}
            elif z == 'J': v_angs = {5:[-0.6, 0.2], 15:[-1.0, 0.0], 30:[-0.5, 0.0], 45:[-0.3, 0.0], 60:[-0.3], 75:[-0.3]}
        else: # Longitudinal
            if z == 'F': v_angs = {5:[-1.6], 15:[-1.3], 30:[-1.1], 45:[-1.1], 60:[-1.1], 75:[-1.1]}
            elif z == 'G': v_angs = {5:[-1.3], 15:[-1.3], 30:[-1.4], 45:[-1.4], 60:[-1.2], 75:[-1.2]}
            elif z == 'H': v_angs = {5:[-0.7], 15:[-0.6], 30:[-0.8], 45:[-0.9], 60:[-0.8], 75:[-0.8]}
            elif z == 'I': v_angs = {5:[-0.6], 15:[-0.5], 30:[-0.5], 45:[-0.5], 60:[-0.5], 75:[-0.5]}
            
        # Logica de interpolacao linear manual
        ang = angulo_graus
        keys = sorted(list(v_angs.keys()))
        
        vals_final = []
        
        # Achar intervalo
        lower = keys[0]
        upper = keys[-1]
        
        if ang <= lower:
            vals_final = v_angs[lower]
        elif ang >= upper:
            vals_final = v_angs[upper]
        else:
            for i in range(len(keys)-1):
                if keys[i] <= ang and ang <= keys[i+1]:
                    lower = keys[i]
                    upper = keys[i+1]
                    break
            
            v_low = v_angs[lower]
            v_up = v_angs[upper]
            
            # Interpolar
            for k in range(len(v_low)):
                y1 = v_low[k]
                # Se a lista superior for menor, usa o ultimo valor
                idx_up = k
                if k >= len(v_up): idx_up = len(v_up) - 1
                y2 = v_up[idx_up]
                
                res = y1 + (ang - lower) * (y2 - y1) / (upper - lower)
                vals_final.append(round(res, 4))
                
        print("  Zona", z, ":", vals_final)

    print("\n")

    # ==========================================
    # PARTE 3: NEVE (NP EN 1991-1-3)
    # ==========================================
    print("--- 3. ACAO DA NEVE ---")
    
    try:
        H_altitude = float(input("Altitude (H) [m]: "))
        zona_neve_str = input("Zona de Neve (1, 2, 3): ")
        zona_neve = int(zona_neve_str)
        
        Ce_str = input("Coef. Exposicao Ce (padrao 1.0): ")
        if Ce_str == "":
            Ce_neve = 1.0
        else:
            Ce_neve = float(Ce_str)
            
    except ValueError:
        print("Erro nos dados da neve.")
        return

    Cz = 0.20
    if zona_neve == 1: Cz = 0.30
    elif zona_neve == 2: Cz = 0.20
    elif zona_neve == 3: Cz = 0.15 # Corrigido para 0.15 conforme Z3 padrao PT

    sk = round(Cz * (1 + (H_altitude / 500)**2), 4)

    mu1 = 0.8
    if angulo_graus > 30 and angulo_graus < 60:
        mu1 = round(0.8 * (60 - angulo_graus) / 30, 4)
    elif angulo_graus >= 60:
        mu1 = 0.0

    Ct_neve = 1.0
    S_neve = round(mu1 * Ce_neve * Ct_neve * sk, 4)

    print("\n---------------------------------------")
    print("RESULTADOS ALINEA 3: NEVE (Z", zona_neve, ")")
    print("---------------------------------------")
    print("Altitude (H)      =", H_altitude, "m")
    print("Coeficiente Cz    =", Cz)
    print("Valor solo (sk)   =", sk, "kN/m2")
    print("Inc. Cobertura    =", angulo_graus, "deg")
    print("Coef. Forma (mu1) =", mu1)
    print("Coef. Exp (Ce)    =", Ce_neve)
    print("--> Carga Neve (S) =", S_neve, "kN/m2")
    print("---------------------------------------\n")

# Executar
resolucao_teste_completa()

   RESOLUCAO TESTE: VENTO + NEVE
Nota: Calculos arredondados a 4 casas.

--- 1. VENTO: DADOS DE ENTRADA ---


Altura do pilar (h) [m]:  6
Vao do portico (L) [m]:  25
Inclinacao cobertura [graus]:  10
Zona de Vento (A ou B):  B



---------------------------------------
RESULTADOS ALINEA 1: q_p(z)
---------------------------------------
Alt. Cumeeira (z)  = 8.2041 m
vb                 = 30.0 m/s

Categorias de Terreno:
1: Zona I (Mar)
2: Zona II (Vegetacao)
3: Zona III (Suburbana)
4: Zona IV (Urbana)


Categoria (1-4):  4



---------------------------------------
RESULTADOS ALINEA 1: q_p(z)
---------------------------------------
Parametro z0       = 1.0 m
Parametro z_min    = 15.0 m
Alt. Calculo (z_e) = 15.0 m
Fator Terreno (kr) = 0.2343
Rugosidade (cr)    = 0.6345
Vel. Media (vm)    = 19.035 m/s
Turbulencia (Iv)   = 0.3693
--> q_p(z)         = 0.8119 kN/m2
---------------------------------------

--- 2. VENTO: COEFICIENTES (c_pe) ---


KeyboardInterrupt: Interrupted by user